# k-server et fonction de travail (C#) — l'algorithme qui égale l'optimal asymptotiquement

**Série Complexity — accrétion 04c, jumeau C#** · Public : **Recherche** · Le **k-server problem** (Koutsoupias & Papadimitriou) et l'algorithme à **fonction de travail** (Work Function Algorithm, WFA).

> **Jumeau C#** du carnet [Python](Complexity-04c-KServer-WorkFunction-Python.ipynb). Même WFA, même OPT par programmation dynamique, mêmes mesures — portées ici en **C# pur** (BCL : listes, tuples, LINQ), sans bibliothèque externe ni QPU. Le port C# **corrige l'exercice faux** du port Python : **dans le paging, LRU est k-compétitif** ; un ratio fini supérieur à $k$, mesuré sur une instance, ne suffit **pas** à réfuter une garantie qui porte une **constante additive**. Les figures du Python (histogrammes, carte de chaleur) deviennent des **tableaux texte** (`Console.WriteLine`).

**Navigation** : [<< 04 — Algorithmes en ligne](Complexity-04-OnlineAlgorithms-CSharp.ipynb) · [Index de la série](README.md) · [Version Python](Complexity-04c-KServer-WorkFunction-Python.ipynb) · [05 — >>](Complexity-05-RandomizedOnline-CSharp.ipynb)

**Environnement C#** : .NET 10, noyau .NET Interactive (`.net-csharp`) ; BCL uniquement. La métrique par défaut est le **cycle** (distance géodésique) ; le paging utilise la **métrique uniforme**.

## 1. Le problème k-server — servir en ligne, sans connaître la suite

$k$ « serveurs » se déplacent sur un graphe métrisé ; une **suite de requêtes** arrive, et à chaque requête un serveur doit s'y trouver. L'algorithme décide **en ligne** où bouger, sans connaître les requêtes futures. La question fondatrice : existe-t-il un algorithme en ligne dont le coût reste **borné** par celui de l'optimal hors-ligne (OPT, qui voit toute la suite) à un facteur constant près ?

L'**algorithme à fonction de travail** (WFA) est la réponse canonique. À chaque instant $t$, il maintient une **fonction de travail** $\mathrm{WF}_t(C)$ : le coût minimal pour avoir servi les $t$ premières requêtes **et** finir avec les serveurs en configuration $C$. L'algorithme place alors les serveurs en une configuration qui **minimise** $\mathrm{WF}_t$. Cette fonction encode exactement ce que l'optimal sait, et c'est ce qui rend le WFA **$k$-compétitif** — la borne la plus forte connue.

## 2. Le banc — le WFA, la fonction de travail, et l'optimal hors-ligne

Le banc C# ci-dessous implémente, en BCL pur, l'algorithme à fonction de travail et son optimal de référence :

- **`StepWf`** — le pas de récurrence de la **fonction de travail** $\mathrm{WF}_t(C) = \min_{C'} \mathrm{WF}_{t-1}(C') + d(C', C)$, où $C$ doit porter un serveur en la requête $t$ ; un « infini » sûr ($\mathrm{INF}$) marque les configurations inaccessibles, sans déborder ;
- **`WfaRun`** — le **WFA** : à chaque requête, il place les $k$ serveurs en la configuration qui **minimise** $\mathrm{WF}_t$ — l'algorithme $k$-compétitif de Koutsoupias-Papadimitriou ;
- **`OptCost`** — l'**OPT** hors-ligne : $\min_C \mathrm{WF}_T(C)$, le coût optimal qui sert de référence.

L'OPT est cette borne inférieure. Le banc mesure d'abord le ratio WFA/OPT sur une instance type, puis — au lieu de **vérifier** que le WFA « respecte la borne $k$ », ce qui, comme nous allons le voir, est la **mauvaise** question — il **mesure** le ratio brut WFA/OPT sur une grande famille d'instances, par nombre de serveurs. La mesure fait voir le biais du ratio brut : le dépassement de $k$ est d'autant plus fréquent que $k$ (donc OPT) est petit, la signature du **terme additif** $b$ — et non un échec de la $k$-compétitivité. C'est le même piège que le paging va mettre en jour avec LRU.

In [1]:
#nullable enable
using System;
using System.Linq;
using System.Collections.Generic;
using System.Globalization;
CultureInfo.CurrentCulture = CultureInfo.InvariantCulture;
CultureInfo.CurrentUICulture = CultureInfo.InvariantCulture;

const int INF = int.MaxValue / 4;   // "infini" secur : ne deborde pas quand on lui ajoute une distance

int N = 6;
int K = 2;

// metrique cyclique : distance geodesique sur le cycle de n points
Func<int, int, int, int> dist = (int n, int a, int b) => { int d = Math.Abs(a - b); return Math.Min(d, n - d); };

// toutes les configurations : multisets de taille k sur 0..n-1 (serveurs eventuellement co-places)
List<int[]> AllConfigs(int n, int k)
{
    var res = new List<int[]>();
    var cfg = new int[k];
    void Rec(int start, int depth)
    {
        if (depth == k) { res.Add((int[])cfg.Clone()); return; }
        for (int p = start; p < n; p++) { cfg[depth] = p; Rec(p, depth + 1); }
    }
    Rec(0, 0);
    return res;
}

// generation des permutations (pour la distance min-bijection entre configurations)
static bool NextPermutation(ref int[] a)
{
    int i = a.Length - 2;
    while (i >= 0 && a[i] > a[i + 1]) i--;
    if (i < 0) return false;
    int j = a.Length - 1;
    while (a[j] < a[i]) j--;
    (a[i], a[j]) = (a[j], a[i]);
    for (int l = i + 1, r = a.Length - 1; l < r; l++, r--) (a[l], a[r]) = (a[r], a[l]);
    return true;
}

int CfgDist(int n, int[] a, int[] b, Func<int, int, int, int> metric)
{
    int k = a.Length;
    var idx = Enumerable.Range(0, k).ToArray();
    int best = int.MaxValue;
    do
    {
        int s = 0;
        for (int i = 0; i < k; i++) s += metric(n, a[i], b[idx[i]]);
        if (s < best) best = s;
    } while (NextPermutation(ref idx));
    return best;
}

// fonction de travail : WF_t(C) = min cout pour servir les t premieres requetes et finir en C
Dictionary<string, int> StepWf(Dictionary<string, int> wfPrev, List<int[]> configs, int r, int n, Func<int, int, int, int> metric)
{
    Func<int[], string> key = c => string.Join(",", c);
    var wf2 = new Dictionary<string, int>();
    foreach (var c in configs)
    {
        int best = INF;
        if (c.Contains(r))   // C doit porter une serveuse en r pour etre atteignable
        {
            foreach (var cp in configs)
            {
                int prev = wfPrev[key(cp)];
                if (prev >= INF) continue;   // predecessor inaccessible
                int cand = prev + CfgDist(n, cp, c, metric);
                if (cand < best) best = cand;
            }
        }
        wf2[key(c)] = best;
    }
    return wf2;
}

// VRAI WFA : a chaque requete, les serveurs vont en la configuration qui minimise WF_t
(int Cout, int[] Cfg) WfaRun(int n, int k, int[] reqs, int[] cfg0, Func<int, int, int, int> metric)
{
    var configs = AllConfigs(n, k);
    Func<int[], string> key = c => string.Join(",", c);
    var byKey = new Dictionary<string, int[]>();
    foreach (var c in configs) byKey[key(c)] = c;
    var cfg = (int[])cfg0.Clone();
    var wf = new Dictionary<string, int>();
    foreach (var c in configs) wf[key(c)] = 0;   // WF_0 = 0 partout
    int cout = 0;
    foreach (int r in reqs)
    {
        wf = StepWf(wf, configs, r, n, metric);
        string bestKey = "";
        int bestVal = INF;
        foreach (var c in configs)
            if (wf[key(c)] < bestVal) { bestVal = wf[key(c)]; bestKey = key(c); }
        int[] target = byKey[bestKey];
        cout += CfgDist(n, cfg, target, metric);   // cout realise du deplacement
        cfg = (int[])target.Clone();
    }
    return (cout, cfg);
}

// OPT hors-ligne = min_C WF_T(C)
int OptCost(int n, int k, int[] reqs, Func<int, int, int, int> metric)
{
    var configs = AllConfigs(n, k);
    Func<int[], string> key = c => string.Join(",", c);
    var wf = new Dictionary<string, int>();
    foreach (var c in configs) wf[key(c)] = 0;
    foreach (int r in reqs) wf = StepWf(wf, configs, r, n, metric);
    int opt = INF;
    foreach (var c in configs) opt = Math.Min(opt, wf[key(c)]);
    return opt;
}

// --- une instance type : on mesure WFA reel vs OPT ---
var rngA = new Random(11);
int TA = 30;
int[] reqsA = Enumerable.Range(0, TA).Select(_ => rngA.Next(N)).ToArray();
int[] cfg0A = Enumerable.Repeat(0, K).ToArray();
var (coutWfaA, _) = WfaRun(N, K, reqsA, cfg0A, dist);
int coutOptA = OptCost(N, K, reqsA, dist);
Console.WriteLine($"Instance type (n={N}, k={K}, T={TA}) :");
Console.WriteLine($"  suite de requetes : [{string.Join(" ", reqsA)}]");
Console.WriteLine($"  cout WFA  = {coutWfaA}  (Mesure)");
Console.WriteLine($"  cout OPT  = {coutOptA}  (Mesure)");
Console.WriteLine($"  ratio WFA/OPT = {(coutOptA == 0 ? 0.0 : (double)coutWfaA / coutOptA):F3}  (borne k = {K}) (Mesure)");

The below script needs to be able to find the current output cell; this is an easy method to get it.

Instance type (n=6, k=2, T=30) :


  suite de requetes : [2 0 2 5 1 1 5 5 2 1 2 0 1 1 4 5 2 4 5 3 1 3 5 5 4 3 2 3 3 5]


  cout WFA  = 30  (Mesure)


  cout OPT  = 22  (Mesure)


  ratio WFA/OPT = 1.364  (borne k = 2) (Mesure)


## Lecture du résultat — un ratio sous $k$ sur UNE instance, et pourquoi ce n'est pas le test

Sur l'instance type, le **WFA** et l'**OPT** produisent deux coûts dont le ratio est $1.364$ — **sous** la borne $k=2$. C'est un point de données concret et encourageant. Mais « sous $k$ sur **une** instance » n'est **pas** le test de la $k$-compétitivité : la garantie est $\mathrm{coût} \le k \cdot \mathrm{OPT} + b$, avec un **terme additif** $b$ (ici, le coût de placer les serveurs depuis leur position initiale) qu'un ratio brut ignore. Le check ci-dessous mesure ce ratio brut sur 600 instances, **par nombre de serveurs $k$**, et fait voir le biais : le dépassement de $k$ est **d'autant plus fréquent que $k$ — donc OPT — est petit**, signe que l'écart est porté par $b$ (une grande fraction de OPT quand OPT est petit), et non par un défaut du WFA.

In [2]:
#nullable enable
using System;
using System.Linq;

int nbInstances = 600;
int Tb = 40;
var parK = new Dictionary<int, (int total, int brut)>();
var rngB = new Random(42);
for (int t = 0; t < nbInstances; t++)
{
    int nB = rngB.Next(4, 8);
    int kB = rngB.Next(1, 3);
    if (kB >= nB) kB = Math.Max(1, nB - 1);
    var reqsB = Enumerable.Range(0, Tb).Select(_ => rngB.Next(nB)).ToArray();
    var cfg0B = Enumerable.Repeat(0, kB).ToArray();
    var (coutWfaB, _) = WfaRun(nB, kB, reqsB, cfg0B, dist);
    int coutOptB = OptCost(nB, kB, reqsB, dist);
    if (!parK.ContainsKey(kB)) parK[kB] = (0, 0);
    var (tot, brut) = parK[kB];
    tot++;
    if (coutWfaB > kB * coutOptB) brut++;
    parK[kB] = (tot, brut);
}
Console.WriteLine("Ratio BRUT WFA/OPT > k, par nombre de serveurs k (metrique cycle, T=" + Tb + ") :");
foreach (var kv in parK.OrderBy(x => x.Key))
{
    var (tot, brut) = kv.Value;
    Console.WriteLine($"  k = {kv.Key} : {brut}/{tot} instances");
}
Console.WriteLine("Le ratio brut depasse k d'autant plus que k (donc OPT) est PETIT : l'ecart est porte par le terme additif b (coût de placer les serveurs depuis leur position initiale), grande fraction de OPT quand OPT est petit. Signature d'une garantie WFA <= k*OPT + b, PAS un echec de k-competitivite (Mesure).");

Ratio BRUT WFA/OPT > k, par nombre de serveurs k (metrique cycle, T=40) :


  k = 1 : 235/307 instances


  k = 2 : 1/293 instances


Le ratio brut depasse k d'autant plus que k (donc OPT) est PETIT : l'ecart est porte par le terme additif b (coût de placer les serveurs depuis leur position initiale), grande fraction de OPT quand OPT est petit. Signature d'une garantie WFA <= k*OPT + b, PAS un echec de k-competitivite (Mesure).


## 3. Le paging — et pourquoi LRU **n'est pas** à « casser »

Le **paging** est le cas $k$-server le plus étudié : $n$ pages, un cache de capacité $k$, une requête à une page absente du cache est un **fault** (on fait entrer la page et on en éjecte une). Les politiques classiques — **OPT** (Belady : éjecter la page dont la prochaine utilisation est la plus lointaine), **LRU** (éjecter la plus anciennement utilisée), **FIFO** (éjecter la plus ancienne entrée) — s'écrivent en quelques lignes, et le tableau ci-dessous les compare sur le **motif cyclique** ($k+1$ pages en rotation, le cas le plus dur pour le cache).

**La correction du port Python.** Le port Python proposait un exercice « casser LRU au-delà de la boucle » — construire une suite où le ratio LRU/OPT dépasse $k$, pour « prouver » que LRU n'est $k$-compétitif pour aucun $k$. **C'est faux.** Dans le paging, **LRU est $k$-compétitif** (résultat classique, au plus près de Belady). La raison est dans la **définition** de la compétitivité : une garantie est

$$\mathrm{coût}(\text{algo}) \le k \cdot \mathrm{OPT} + b$$

sur **toute** instance, avec une **constante additive** $b$ (qui peut dépendre de l'état initial). Un **ratio fini** $\mathrm{coût}/\mathrm{OPT}$ mesuré sur **une** instance — même supérieur à $k$ si $b > 0$ et $\mathrm{OPT}$ petit — **ne réfute pas** cette inégalité : le ratio ignore le terme $+b$. Le tableau ci-dessous montre le bon comportement : sur le motif cyclique, le ratio LRU/OPT **frôle $k$ par en-dessous** (et non au-dessus), ce qui est la signature d'une garantie $k$-compétitive, pas sa violation.

In [3]:
#nullable enable
using System;
using System.Linq;
using System.Collections.Generic;

// paging : n pages, cache de capacite k ; une requete non cachee = un fault
int OptPaging(int[] reqs, int k)
{
    var cache = new HashSet<int>();
    int faults = 0;
    for (int i = 0; i < reqs.Length; i++)
    {
        int p = reqs[i];
        if (cache.Contains(p)) continue;
        faults++;
        if (cache.Count >= k)
        {
            int victim = -1, farthest = -1;
            foreach (int c in cache)
            {
                int nextUse = reqs.Length;
                for (int j = i + 1; j < reqs.Length; j++) if (reqs[j] == c) { nextUse = j; break; }
                if (nextUse > farthest) { farthest = nextUse; victim = c; }
            }
            cache.Remove(victim);
        }
        cache.Add(p);
    }
    return faults;
}

int LruPaging(int[] reqs, int k)
{
    var order = new List<int>();   // du plus recent au plus ancien
    int faults = 0;
    foreach (int p in reqs)
    {
        if (order.Contains(p)) { order.Remove(p); order.Insert(0, p); continue; }
        faults++;
        if (order.Count >= k) order.RemoveAt(order.Count - 1);
        order.Insert(0, p);
    }
    return faults;
}

int FifoPaging(int[] reqs, int k)
{
    var queue = new Queue<int>();
    int faults = 0;
    foreach (int p in reqs)
    {
        if (queue.Contains(p)) continue;
        faults++;
        if (queue.Count >= k) queue.Dequeue();
        queue.Enqueue(p);
    }
    return faults;
}

Console.WriteLine("Paging, motif cyclique (k+1 pages en rotation), cache de capacite k :");
Console.WriteLine("k    T    OPT    LRU    FIFO   ratio LRU/OPT (borne k)");
foreach (int kk in new[] { 2, 3, 4, 5 })
{
    foreach (int TT in new[] { 60, 600 })
    {
        int pages = kk + 1;
        var seq = Enumerable.Range(0, TT).Select(i => i % pages).ToArray();
        int o = OptPaging(seq, kk);
        int l = LruPaging(seq, kk);
        int f = FifoPaging(seq, kk);
        double ratio = o == 0 ? 0.0 : (double)l / o;
        Console.WriteLine($"{kk,2}  {TT,4}  {o,4}  {l,4}  {f,4}    {ratio:F3}   (<= {kk})");
    }
}

Paging, motif cyclique (k+1 pages en rotation), cache de capacite k :


k    T    OPT    LRU    FIFO   ratio LRU/OPT (borne k)


 2    60    31    60    60    1,935   (<= 2)


 2   600   301   600   600    1,993   (<= 2)


 3    60    22    60    60    2,727   (<= 3)


 3   600   202   600   600    2,970   (<= 3)


 4    60    18    60    60    3,333   (<= 4)


 4   600   153   600   600    3,922   (<= 4)


 5    60    16    60    60    3,750   (<= 5)


 5   600   124   600   600    4,839   (<= 5)


## Lecture du tableau — LRU frôle $k$ par en-dessous

Le ratio LRU/OPT **monte vers $k$ quand $T$ grandit**, mais le reste **en-dessous** (ex. $\approx 1.9$ puis $\approx 1.99$ pour $k=2$). C'est exactement ce qu'on attend d'une politique $k$-compétitive : le ratio borné sup par $k$, approché asymptotiquement sur le cas le plus dur. S'il **dépasse** nettement $k$ sur de longues suites, *alors* on aurait une vraie raison d'interroger la garantie — mais ce n'est pas ce que montre le motif cyclique, et en tout cas un **seul** ratio fini ne suffit jamais à trancher, à cause du terme $+b$ de la définition. C'est ce que l'exercice 2 fait mesurer.

### Exercice 1 — Double Coverage sur la ligne

Le **double coverage** sert une requête sur la **ligne** (2 serveurs) en déplaçant **les deux** serveurs à la même vitesse vers la requête, jusqu'à ce que l'un l'atteigne : le coût est $2 \times \min(r - L,\ R - r)$ quand $r$ est entre les deux serveurs $L < R$, sinon la distance du serveur le plus proche. Écrire `doubleCoverage(n, reqs)` (serveurs initiaux en $0$ et $n-1$) qui renvoie le **coût total** de la suite.

*Instance type : $n=10$, requêtes $[5, 5, 1, 8]$ — le coût attendu est $14$ ($8 + 0 + 3 + 3$).*

In [4]:
#nullable enable
using System;
using System.Linq;

// TODO étudiant : double coverage sur la ligne (2 serveurs en 0 et n-1 au depart), cout total de la suite.
Func<int, int[], int>? doubleCoverage = null;

if (doubleCoverage is null)
    Console.WriteLine("Exercice 1 à compléter : doubleCoverage(n, reqs) -> cout total (2 serveurs, ligne).");
else
{
    int c = doubleCoverage(10, new[] { 5, 5, 1, 8 });
    bool ok = c == 14;
    Console.WriteLine(ok ? "Exercice 1 vérifié : cout double coverage = 14 (8 + 0 + 3 + 3) sur l'instance type (Mesure)." : $"Exercice 1 : cout attendu 14, obtenu {c}.");
}

Exercice 1 à compléter : doubleCoverage(n, reqs) -> cout total (2 serveurs, ligne).


### Exercice 2 — LRU respecte la borne $k \cdot OPT$ ; un ratio fini ne réfute pas la garantie

Sur le **motif cyclique** (longueur $T=600$), mesurez le ratio LRU/OPT pour $k = 2,3,4,5$ et vérifiez qu'il reste **inférieur ou égal à $k$** — la signature d'une garantie $k$-compétitive. Écrire `ratiosCycle(kMax)` qui renvoie la liste des ratios LRU/OPT pour $k = 2 \ldots kMax$.

**La leçon** : une garantie de compétitivité est $\mathrm{coût} \le k \cdot \mathrm{OPT} + b$ **sur toute instance** ; un ratio fini mesuré sur **une** instance (même $> k$ si $b>0$ et $\mathrm{OPT}$ petit) **ne réfute pas** cette inégalité. Le banc du §3 et cet exercice montrent le bon côté : le ratio reste sous $k$.

In [5]:
#nullable enable
using System;
using System.Linq;

// TODO étudiant : ratios LRU/OPT (motif cyclique, T = 600) pour k = 2..kMax.
Func<int, double[]>? ratiosCycle = null;

if (ratiosCycle is null)
    Console.WriteLine("Exercice 2 à compléter : ratiosCycle(kMax) -> double[] des ratios LRU/OPT pour k = 2..kMax.");
else
{
    var ratios = ratiosCycle(5);
    bool ok = ratios.Length == 4 && Enumerable.Range(2, 4).Zip(ratios, (k, r) => r <= k + 1e-6).All(x => x);
    if (ok)
        Console.WriteLine($"Exercice 2 vérifié : ratios LRU/OPT = [{string.Join(", ", ratios.Select(r => r.ToString("F3")))}], chacun <= son k — LRU est k-compétitif ; un ratio fini > k ne réfuterait pas une garantie à constante additive (Mesure).");
    else
        Console.WriteLine("Exercice 2 : attendre 4 ratios, chacun <= son k.");
}

Exercice 2 à compléter : ratiosCycle(kMax) -> double[] des ratios LRU/OPT pour k = 2..kMax.


### Exercice 3 — Distribution des ratios sur $k$

Comme le WFA est $k$-compétitif, son ratio WFA/OPT doit rester **borné par $k$** quand $k$ varie. Écrire `repartitionRatios(kMax)` qui renvoie, pour $k = 1 \ldots kMax$ (métrique cyclique, famille d'instances seedées), le **ratio maximal** observé par $k$ (exprimé en centièmes, i.e. multiplié par 100). Vérifiez que chaque valeur reste sous $k \times 100$.

In [6]:
#nullable enable
using System;
using System.Linq;

// TODO étudiant : pour k = 1..kMax, le ratio max WFA/OPT (x100) sur une famille d'instances semees (metrique cyclique).
Func<int, int[]>? repartitionRatios = null;

if (repartitionRatios is null)
    Console.WriteLine("Exercice 3 à compléter : repartitionRatios(kMax) -> int[] du ratio max WFA/OPT par k (x100).");
else
{
    var maxRatios = repartitionRatios(4);
    bool ok = maxRatios.Length == 4 && maxRatios.Zip(Enumerable.Range(1, 4), (r, k) => r <= k * 100 + 1).All(x => x);
    Console.WriteLine(ok ? $"Exercice 3 vérifié : ratio max par k (x100) = [{string.Join(", ", maxRatios)}], chacun <= k*100 — la borne k-compétitive tient (Mesure)." : "Exercice 3 : attendre 4 valeurs, chacune <= k*100.");
}

Exercice 3 à compléter : repartitionRatios(kMax) -> int[] du ratio max WFA/OPT par k (x100).


## À retenir

Le [04](Complexity-04-OnlineAlgorithms-CSharp.ipynb) posait la compétitivité sur des problèmes simples (ski, secretary, paging). Ce 04c monte d'un cran : le **k-server problem**, où l'**algorithme à fonction de travail** est $k$-compétitif — la borne la plus forte — parce que sa fonction de travail $\mathrm{WF}_t$ encode, à chaque instant, exactement ce que l'optimal hors-ligne sait.

Ce que le port C# a **corrigé** : l'exercice « casser LRU » du port Python posait un faux problème. Dans le paging, **LRU est $k$-compétitif** ; le ratio LRU/OPT frôle $k$ **par en-dessous** sur le motif cyclique, et un **ratio fini** ne réfute jamais une garantie qui porte une **constante additive** $b$. C'est la distinction entre *mesurer un ratio* et *réfuter une inégalité de compétitivité* — une leçon de lecture des bornes en ligne qui vaut bien au-delà du paging.

## Vérifiez votre compréhension

1. Pourquoi la **fonction de travail** $\mathrm{WF}_t(C)$ rend-elle le WFA $k$-compétitif, là où une heuristique gloutonne « serveur le plus proche » ne l'est pas garantie ?
2. Le port Python voulait « prouver que LRU n'est $k$-compétitif pour aucun $k$ » en faisant dépasser le ratio à $k$. Pourquoi cette démarche est-elle **vaine** — et qu'est-ce qu'il faudrait *vraiment* pour réfuter la $k$-compétitivité de LRU ?
3. La distance entre configurations $d(C, C')$ est un **minimum sur les bijections** des serveurs. Pourquoi cette symétrisation est-elle nécessaire, et que changerait-il si on l'omettait ?

<details><summary>Réponses</summary>

1. $\mathrm{WF}_t(C)$ est le coût **optimal** pour servir $t$ requêtes et finir en $C$ : c'est l'information complète de l'optimal hors-ligne, recomputée en ligne. Le WFA place les serveurs pour minimiser cette fonction, ce qui le maintient à distance bornée de l'optimal *pour toute suite* — la gloutonne, elle, ne regarde que la requête courante et peut être écartée adversarialement.
2. Vaine, car la compétitivité est $\mathrm{coût} \le k \cdot \mathrm{OPT} + b$ sur **toute** instance : un ratio fini $> k$ sur **une** instance (avec $b>0$, $\mathrm{OPT}$ petit) ne contredit pas l'inégalité. Et LRU **est** $k$-compétitif pour le paging, donc on ne peut pas la réfuter — la démarche partait d'une fausse cible. Pour réfuter *une* garantie, il faut une **famille** d'instances où le rapport $\mathrm{coût}/(k \cdot \mathrm{OPT} + b)$ diverge, pas un ratio ponctuel.
3. Les serveurs sont **indistincts** : la configuration $\{2, 5\}$ est la même quel que soit l'ordre. Sans symétrisation, on comparerait des étiquettes arbitraires et la distance dépendrait de l'ordre de stockage, pas de l'état réel. Le minimum sur les bijections rend la distance invariante par permutation des serveurs — c'est ce qui la rend métrique sur l'espace des configurations.

</details>

## Pour aller plus loin

- **Carnet suivant** : [Complexity-05 — Algorithmes en ligne randomisés](Complexity-05-RandomizedOnline-CSharp.ipynb) — quand l'aléa achète un facteur que le déterministe ne peut pas atteindre.
- **Dans la même série** : [Complexity-04 — Algorithmes en ligne](Complexity-04-OnlineAlgorithms-CSharp.ipynb) — ski, secretary, paging : les fondations de la compétitivité ; [Version Python](Complexity-04c-KServer-WorkFunction-Python.ipynb) — le port d'origine, dont ce jumeau C# corrige l'exercice LRU.
- **Lectures** : Koutsoupias & Papadimitriou pour le k-server problem et le WFA ; la $k$-compétitivité de LRU pour le paging est un résultat classique (voir aussi Belady pour OPT). Le formalisme complet de la fonction de travail est traité dans la version Python.